In [1]:
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModel
from model_utils import pad_and_read_bert

In [2]:
test_token_ids = [[104, 29262, 12, 180, 12, 279, 40069, 2839, 495, 26789, 347, 9056, 1891, 14124, 354, 8310, 7904, 6199, 1116, 20541, 3624, 154, 7109, 22197, 415, 102, 5320, 7381, 1054, 2463, 34806, 8005, 179, 44344, 4], [133, 5320, 7381, 1054, 9497, 42961, 29096, 19203, 10010, 11990, 7078, 260, 271, 7110, 179, 102, 4006, 12, 180, 12, 279, 14582, 20541, 3624, 11173, 4, 5320, 7381, 1054, 9497, 20364, 24021, 23886, 29096, 5606, 12, 180, 12, 279, 40069, 2839, 495, 26789, 347, 9056, 1891, 14124, 7325, 271, 7110, 196, 13837, 698, 35, 1978, 102, 4, 119, 4, 19203, 463, 18716, 5632, 7109, 11432, 29, 1116, 9502, 5743, 20541, 3624, 4], [5320, 7381, 1054, 24848, 261, 19203, 271, 7110, 196, 102, 2466, 12, 180, 12, 279, 6, 42985, 2368, 1873, 7760, 179, 627, 44344, 16424, 1033, 1116, 7109, 26840, 22197, 415, 102, 18076, 2463, 34806, 8490, 6, 4345, 260, 11408, 2787, 4006, 12, 180, 1630, 42317, 1990, 627, 28923, 13776, 108, 27505, 28694, 4], [250, 876, 12, 23589, 7333, 35507, 18716, 179, 627, 44344, 9996, 857, 1033, 1116, 7109, 22197, 415, 260, 2463, 34806, 22799, 179, 5320, 7381, 1054, 6, 4297, 11515, 2192, 28357, 627, 29, 687, 13771, 354, 34664, 24483, 5632, 102, 42985, 337, 1873, 1825, 4], [250, 261, 16093, 876, 12, 23589, 7333, 35507, 18716, 179, 627, 44344, 9996, 857, 1033, 1116, 7109, 22197, 415, 260, 2463, 34806, 22799, 179, 5320, 7381, 1054, 6, 4297, 11515, 2192, 28613, 627, 29, 687, 13771, 354, 34664, 24483, 5632, 102, 42985, 337, 1873, 1825, 4], [713, 24863, 6, 4006, 12857, 36600, 6, 5320, 7381, 1054, 24848, 11990, 271, 7110, 196, 463, 18716, 40069, 2839, 495, 26789, 347, 9056, 1891, 14124, 6, 1580, 5606, 6, 5632, 7109, 11432, 29, 1116, 9502, 12, 5743, 20541, 3624, 4], [5320, 7381, 1054, 9497, 9226, 24863, 271, 7110, 196, 40069, 2839, 495, 26789, 347, 9056, 1891, 14124, 6, 1116, 5320, 7381, 1054, 6, 261, 9502, 12, 5743, 20541, 3624, 30364, 179, 627, 44344, 9996, 857, 1033, 1116, 7109, 22197, 415, 18522, 18, 38136, 347, 26930, 39948, 4], [250, 2466, 12, 180, 12, 279, 7760, 18716, 5632, 34549, 7109, 26840, 22197, 179, 102, 5320, 7381, 1054, 2463, 34806, 22799, 4006, 12857, 6643, 7325, 29847, 19010, 428, 3760, 6, 11515, 2192, 28613, 19203, 6, 13437, 8877, 354, 417, 4048, 7761, 30704, 2596, 463, 12690, 417, 1496, 3175, 463, 26111, 179, 102, 298, 16497, 2463, 33492, 13005, 4], [133, 5320, 7381, 1054, 9497, 42961, 6677, 4208, 102, 13447, 16459, 9226, 10669, 20466, 560, 44547, 2629, 12406, 18541, 12473, 102, 14582, 20541, 3624, 1644, 179, 44344, 4]]

In [3]:
bert_model = AutoModel.from_pretrained("roberta-large")

Some weights of the model checkpoint at roberta-large were not used when initializing RobertaModel: ['lm_head.layer_norm.bias', 'lm_head.bias', 'lm_head.layer_norm.weight', 'lm_head.dense.weight', 'lm_head.dense.bias']
- This IS expected if you are initializing RobertaModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [4]:
# def pad_and_read_bert(bert_token_ids, bert_model):
#     length = np.array([len(d) for d in bert_token_ids])
#     max_length = max(length)

#     if max_length > 512:
#         raise ValueError('Error! Segment too long!')

#     device = bert_model.device
#     print("Type token ids",type(bert_token_ids) )
#     print(bert_token_ids)
#     docs = torch.tensor([doc + [0] * (max_length - len(doc)) for doc in bert_token_ids], device=device)
#     attention_masks = torch.tensor([[1] * len(doc) + [0] * (max_length - len(doc)) for doc in bert_token_ids], device=device)
#     with torch.no_grad():
#         output = bert_model(docs, attention_masks)
   
#     return output.last_hidden_state , length

In [5]:
output,length = pad_and_read_bert(test_token_ids,bert_model)

In [6]:
len(test_token_ids[1])

72

In [7]:
output

tensor([[[-0.0853,  0.0049, -0.1253,  ...,  0.1263,  0.0412,  0.2274],
         [-0.1048, -0.0298, -0.1272,  ...,  0.0159, -0.0093,  0.1877],
         [ 0.0899,  0.0060, -0.2005,  ...,  0.1495,  0.1843,  0.1671],
         ...,
         [-0.0867,  0.0075, -0.1268,  ...,  0.1234,  0.0442,  0.2278],
         [-0.0867,  0.0077, -0.1267,  ...,  0.1233,  0.0443,  0.2277],
         [-0.0863,  0.0067, -0.1261,  ...,  0.1245,  0.0429,  0.2278]],

        [[-0.1702, -0.0138,  0.0054,  ...,  0.1523, -0.0770,  0.1601],
         [-0.2580,  0.2715, -0.5193,  ...,  0.0981,  0.0838,  0.0316],
         [ 0.1049,  0.1727, -0.2844,  ...,  0.3270,  0.7224, -0.0475],
         ...,
         [-0.1738,  0.4478, -0.0980,  ..., -0.2222, -0.3035, -0.1920],
         [-0.0600,  0.1247,  0.0878,  ...,  0.3922, -0.3866, -0.7211],
         [ 0.0982,  0.0337, -0.0311,  ..., -0.0512,  0.1337, -0.0321]],

        [[-0.2525, -0.0347, -0.1696,  ...,  0.1669, -0.0316,  0.1327],
         [-0.2039,  0.1561, -0.2014,  ..., -0